In [1]:
import numpy as np
import plotly.graph_objects as go
import plotly.io as pio

pio.renderers.default = "browser"


def get_circumcircle(p1, p2, p3):
    x1, y1 = p1; x2, y2 = p2; x3, y3 = p3
    D = 2 * (x1 * (y2 - y3) + x2 * (y3 - y1) + x3 * (y1 - y2))
    if abs(D) < 1e-9: return None, 0
    ux = ((x1**2 + y1**2) * (y2 - y3) + (x2**2 + y2**2) * (y3 - y1) + (x3**2 + y3**2) * (y1 - y2)) / D
    uy = ((x1**2 + y1**2) * (x3 - x2) + (x2**2 + y2**2) * (x1 - x3) + (x3**2 + y3**2) * (x2 - x1)) / D
    center = np.array([ux, uy])
    radius = np.linalg.norm(center - p1)
    return center, radius

def manual_delaunay(points):
    min_p, max_p = np.min(points, axis=0) - 1, np.max(points, axis=0) + 1
    p1, p2, p3 = [min_p[0], min_p[1]], [max_p[0]+5, min_p[1]], [min_p[0], max_p[1]+5]
    all_pts = np.vstack([points, p1, p2, p3])
    triangles = [[len(points), len(points)+1, len(points)+2]]
    
    for i, p in enumerate(points):
        bad_triangles = []
        for tri in triangles:
            center, radius = get_circumcircle(all_pts[tri[0]], all_pts[tri[1]], all_pts[tri[2]])
            if center is not None and np.linalg.norm(center - p) < radius:
                bad_triangles.append(tri)
        
        polygon = []
        for tri in bad_triangles:
            for j in range(3):
                edge = tuple(sorted((tri[j], tri[(j+1)%3])))
                if sum(1 for t in bad_triangles if edge[0] in t and edge[1] in t) == 1:
                    polygon.append(edge)
        
        triangles = [t for t in triangles if t not in bad_triangles]
        for edge in polygon:
            triangles.append([edge[0], edge[1], i])
    
    return np.array([t for t in triangles if not any(v >= len(points) for v in t)])


def f(x, y, z): return (x**2)/2 + (y**2)/3 + (z**2)/4 - 15

grid_size = 18
x = np.linspace(-10, 10, grid_size)
y = np.linspace(-10, 10, grid_size)
z = np.linspace(-10, 10, grid_size)
Xg, Yg, Zg = np.meshgrid(x, y, z, indexing="ij")
Fg = f(Xg, Yg, Zg)

cube_edges = []
cube_size = x[1] - x[0]
rng = np.random.default_rng(0)

def make_cube_lines(x0, y0, z0, size):
    dx = [0, size]; lines = []
    for sx in dx:
        for sy in dx: lines.append(([x0+sx,x0+sx],[y0+sy,y0+sy],[z0,z0+size]))
        for sz in dx: lines.append(([x0+sx,x0+sx],[y0,y0+size],[z0+sz,z0+sz]))
        for sz in dx: lines.append(([x0,x0+size],[y0+sy,y0+sy],[z0+sz,z0+sz]))
    return lines

for i in range(grid_size-1):
    for j in range(grid_size-1):
        for k in range(grid_size-1):
            if np.min(Fg[i:i+2,j:j+2,k:k+2])*np.max(Fg[i:i+2,j:j+2,k:k+2]) < 0:
                if rng.random() < 0.8:
                    cube_edges.extend(make_cube_lines(x[i],y[j],z[k],cube_size))


a, b, c = np.sqrt(30), np.sqrt(45), np.sqrt(60)
theta = np.linspace(0, 2*np.pi, 60); phi = np.linspace(0, np.pi, 40)
T, P = np.meshgrid(theta, phi)


sub_size = 15
T_sub = T[:sub_size, :sub_size].ravel()
P_sub = P[:sub_size, :sub_size].ravel()
points_2d_sub = np.vstack([T_sub, P_sub]).T


faces_sub = manual_delaunay(points_2d_sub)


X_sub = a * np.sin(P_sub) * np.cos(T_sub)
Y_sub = b * np.sin(P_sub) * np.sin(T_sub)
Z_sub = c * np.cos(P_sub)


fig = go.Figure()


for lx,ly,lz in cube_edges:
    fig.add_trace(go.Scatter3d(x=lx, y=ly, z=lz, mode="lines",
                               line=dict(color="rgba(150,150,150,0.3)", width=1), showlegend=False))


fig.add_trace(go.Scatter3d(x=(a*np.sin(P)*np.cos(T)).ravel(), y=(b*np.sin(P)*np.sin(T)).ravel(), 
                           z=(c*np.cos(P)).ravel(), mode="markers", 
                           marker=dict(size=1, color="black", opacity=0.2), name="All Points"))


fig.add_trace(go.Mesh3d(
    x=X_sub, y=Y_sub, z=Z_sub,
    i=faces_sub[:,0], j=faces_sub[:,1], k=faces_sub[:,2],
    color="#BE3144", opacity=0.9, name="Manual Delaunay Patch"
))

fig.update_layout(title=" իմպլեմենտացիա միայն մի հատվածի վրա",
                  scene=dict(xaxis_title="X", yaxis_title="Y", zaxis_title="Z"))
fig.show()

print(f"Հատվածի կետերի քանակը = {len(X_sub)}")
print(f"Ձեռքով ստացված եռանկյունների քանակը = {len(faces_sub)}")

Հատվածի կետերի քանակը = 225
Ձեռքով ստացված եռանկյունների քանակը = 392
